# FastSurfer regional volumes for the 83 scans (GPU)
Settings: Accelerator **GPU T4**, Internet **On**. Run with Save & Run All (Commit); it takes roughly 4 hours (about 3 minutes per scan).
Output: `/kaggle/working/pd_volumes/volumes.csv` (83 rows), plus one stats file and one segmentation per scan.
FastSurfer version used: 2.7.0.dev0 (commit 145c79b). The cell below clones the current repository; pin that commit if you need an exact match.
Segmentation-only mode, 1 mm voxels, corpus callosum, cerebellum and hypothalamus modules disabled.

In [ ]:
# 1. Rebuild the raw scans and the master table
!pip -q install boto3 nibabel
import os, glob, re, tarfile, boto3, pandas as pd
from botocore import UNSIGNED
from botocore.config import Config

BASE, RAW = "/kaggle/working/pd_project", "/tmp/pd_raw"
os.makedirs(BASE, exist_ok=True); os.makedirs(RAW, exist_ok=True)
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
for cohort in ["neurocon", "taowu"]:
    arc = f"/tmp/{cohort}.tar.gz"
    if not os.path.exists(arc):
        s3.download_file("fcp-indi", f"data/Projects/INDI/umf_pd/{cohort}.tar.gz", arc)
    dest = f"{RAW}/{cohort}"
    if not glob.glob(f"{dest}/**/anat/*T1w.nii*", recursive=True):
        os.makedirs(dest, exist_ok=True)
        with tarfile.open(arc, "r:gz") as tar:
            members = [m for m in tar.getmembers() if m.isfile() and
                       ("t1w" in m.name.lower() or m.name.endswith((".tsv", ".csv", ".txt")))]
            tar.extractall(path=dest, members=members, filter="data")
rows = []
for cohort in ["neurocon", "taowu"]:
    for path in sorted(glob.glob(f"{RAW}/{cohort}/**/anat/*T1w.nii*", recursive=True)):
        sid = re.search(r"(sub-[^/_]+)", path).group(1)
        grp = re.search(r"sub-(control|patient)", sid).group(1)
        rows.append(dict(cohort=cohort, subject=sid, path=path, label=0 if grp == "control" else 1))
df = pd.DataFrame(rows)
print(df.groupby(["cohort", "label"]).size())
assert len(df) == 83
df.to_csv(f"{BASE}/master_table.csv", index=False)

In [ ]:
# 2. Install FastSurfer without touching the preinstalled torch, then pin the torch pair that worked
!git clone -q --depth 1 https://github.com/Deep-MI/FastSurfer.git /tmp/FastSurfer
!grep -vE '^(torch|torchvision)([=<>~! ;]|$)|^--' /tmp/FastSurfer/requirements.txt > /tmp/req_no_torch.txt
!pip -q install -r /tmp/req_no_torch.txt 2>&1 | tail -3
!pip -q install --force-reinstall torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -3

In [ ]:
# 3. Fail fast if torch, torchvision or the GPU is not usable
import subprocess
out = subprocess.run(["python3", "-c",
    "import torch, torchvision; from torchvision import transforms; "
    "print(torch.__version__, torchvision.__version__, torch.cuda.is_available())"],
    capture_output=True, text=True)
print(out.stdout, out.stderr[-600:])
assert out.returncode == 0 and out.stdout.strip().endswith("True"), "torch/torchvision/GPU problem"

In [ ]:
# 4. Segment every scan (resumable), then build volumes.csv
import re, shutil, time
from tqdm import tqdm

FS, SD = "/tmp/FastSurfer", "/tmp/fs_out"             # scratch, not saved
OUTD = "/kaggle/working/pd_volumes"                   # saved as Output
for d in (f"{OUTD}/stats", f"{OUTD}/seg"): os.makedirs(d, exist_ok=True)

help_txt = subprocess.run(["./run_fastsurfer.sh", "--help"], cwd=FS, capture_output=True, text=True).stdout
SKIP = [f for f in sorted(set(re.findall(r"--no_[a-z_]+", help_txt))) if re.search(r"cc|cereb|hypo|hyp|callos", f)]
print("skipping modules:", SKIP)

def parse_stats(path):
    meas, vols = {}, {}
    for line in open(path):
        m = re.match(r"# Measure (\w+), \w+, .*?, ([\d.]+), mm\^3", line)
        if m: meas[m.group(1)] = float(m.group(2)); continue
        if line.startswith("#") or not line.strip(): continue
        p = line.split(); vols[p[4]] = float(p[3])
    return meas, vols

def clean_up(d):
    try:
        shutil.rmtree(f"{d}/surf", ignore_errors=True)
        for f in glob.glob(f"{d}/mri/*"):
            if os.path.isdir(f): shutil.rmtree(f, ignore_errors=True)
            else: os.remove(f)
    except Exception as e:
        print("  (cleanup warning:", e, ")")

for _, r in tqdm(list(df.iterrows()), total=len(df)):
    key = f"{r.cohort}_{r.subject}"
    if os.path.exists(f"{OUTD}/stats/{key}.stats"): continue
    t0 = time.time()
    cmd = ["./run_fastsurfer.sh", "--t1", os.path.abspath(r.path), "--sid", key, "--sd", SD,
           "--seg_only", "--allow_root", "--vox_size", "1", *SKIP]
    res = subprocess.run(cmd, cwd=FS, capture_output=True, text=True)
    stats = f"{SD}/{key}/stats/aseg+DKT.stats"
    if not os.path.exists(stats): stats = f"{SD}/{key}/stats/aseg+DKT.VINN.stats"
    seg = f"{SD}/{key}/mri/aparc.DKTatlas+aseg.deep.mgz"
    if res.returncode != 0 or not os.path.exists(stats) or not os.path.exists(seg):
        print("FAILED", key, res.stdout[-500:], res.stderr[-300:]); continue
    shutil.copy(os.path.realpath(stats), f"{OUTD}/stats/{key}.stats")
    shutil.copy(seg, f"{OUTD}/seg/{key}.mgz")
    clean_up(f"{SD}/{key}")
    print(f"{key}: {time.time() - t0:.0f} s", flush=True)

rows = []
for f in sorted(glob.glob(f"{OUTD}/stats/*.stats")):
    key = os.path.basename(f)[:-6]; cohort, sid = key.split("_", 1)
    meas, vols = parse_stats(f)
    rows.append(dict(subject=key, cohort=cohort, sid=sid, **meas, **vols))
vol = pd.DataFrame(rows).merge(
    df[["cohort", "subject", "label"]].rename(columns={"subject": "sid"}), on=["cohort", "sid"])
vol.to_csv(f"{OUTD}/volumes.csv", index=False)
print(vol.shape, "(want 83 rows)")
print(vol.groupby(["cohort", "label"]).size())